In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


**PART 1 - INSERT Exercises**

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(110,'Arjun Nair','Chennai','Mobile',1,30000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(112,'Divya Menon','Kochi','Mobile',1,27000,'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(113,'Farhan Ali','Delhi','Headphones',5,2500,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(114,'Sanjay Gupta','Hyderabad','Tablet',1,26000,'Placed'),
(115,'Neha Singh','Hyderabad','Mobile',2,30000,'Placed'),
(116,'Ravi Kumar','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
3,3


**Part 2 — UPDATE**

In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Shipped' WHERE order_id = 101;

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET quantity = quantity + 1 WHERE order_id = 102;


num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET price = 78000 WHERE product = 'Laptop';

num_affected_rows
5


In [0]:
%sql
UPDATE ecommerce_orders SET city = 'Secunderabad' WHERE customer_name = 'Amit Sharma';

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Delivered' WHERE product = 'Mobile';

num_affected_rows
5


In [0]:
%sql
UPDATE ecommerce_orders SET price = 2500 WHERE product = 'Headphones';

num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET price = price + 1000 WHERE product = 'Tablet';

num_affected_rows
3


In [0]:
%sql
UPDATE ecommerce_orders SET order_status = 'Processing' WHERE city = 'Bangalore';


num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET quantity = 2 WHERE quantity = 1;

num_affected_rows
11


In [0]:
%sql
UPDATE ecommerce_orders SET city = 'Surat' WHERE city = 'Ahmedabad';

num_affected_rows
1


**Part 3 — DELETE**

In [0]:
%sql
DELETE FROM ecommerce_orders WHERE order_status = 'Cancelled';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE quantity > 3;

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE product = 'Camera';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE city = 'Kolkata';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE price < 5000;

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE customer_name LIKE 'A%';

num_affected_rows
2


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE product = 'Tablet';

num_affected_rows
2


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE city = 'Mumbai' AND quantity = 1;

num_affected_rows
0


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE price > 80000;

num_affected_rows
0


In [0]:
%sql
DELETE FROM ecommerce_orders
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


**Part 4 — MERGE / UPSERT**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id, customer_name, city, product, quantity, price, order_status);

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
  target.customer_name = source.customer_name,
  target.city = source.city,
  target.product = source.product,
  target.quantity = source.quantity,
  target.price = source.price,
  target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
VALUES (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
%sql
SELECT * FROM ecommerce_orders
ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
112,Divya Menon,Kochi,Mobile,2,27000,Delivered
115,Neha Singh,Hyderabad,Mobile,2,30000,Delivered


In [0]:
%sql
SELECT 
  source.*,
  CASE 
    WHEN target.order_id IS NOT NULL THEN 'UPDATED'
    ELSE 'INSERTED'
  END AS merge_action
FROM incoming_orders source
LEFT JOIN ecommerce_orders target
ON source.order_id = target.order_id;

order_id,customer_name,city,product,quantity,price,order_status,merge_action
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered,UPDATED
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped,UPDATED
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed,UPDATED
110,Divya Menon,Kochi,Mobile,1,27000,Placed,UPDATED
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed,UPDATED


In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED AND source.quantity > target.quantity THEN
UPDATE SET target.quantity = source.quantity

WHEN NOT MATCHED THEN
INSERT *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING (
  SELECT * FROM incoming_orders
  WHERE order_status <> 'Cancelled'
) AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED AND source.product = 'Laptop' THEN
INSERT *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED AND source.price > target.price THEN
UPDATE SET target.price = source.price

WHEN NOT MATCHED THEN
INSERT *
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
SELECT * FROM ecommerce_orders;

order_id,customer_name,city,product,quantity,price,order_status
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
112,Divya Menon,Kochi,Mobile,2,27000,Delivered
115,Neha Singh,Hyderabad,Mobile,2,30000,Delivered
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


**PART 5 — Realistic UPSERT Scenario**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS 
SELECT * FROM VALUES 
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'), (106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'), 
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed') 
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING daily_updates AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
  target.quantity = source.quantity,
  target.order_status = source.order_status,
  target.price = source.price

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
VALUES (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
%sql
SELECT COUNT(*) AS rows_to_be_updated
FROM daily_updates source
INNER JOIN ecommerce_orders target
ON source.order_id = target.order_id;

rows_to_be_updated
3


In [0]:
%sql
SELECT COUNT(*) AS rows_to_be_inserted
FROM daily_updates source
LEFT ANTI JOIN ecommerce_orders target
ON source.order_id = target.order_id;

rows_to_be_inserted
0


In [0]:
%sql
SELECT *
FROM ecommerce_orders
ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
115,Neha Singh,Hyderabad,Mobile,2,30000,Delivered
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Divya Menon,Kochi,Mobile,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
